# Episode 12: The Conversation Adapter

`consulting` is one question, one answer. But sometimes two agents need to genuinely *talk* — back and forth, no fixed turns.

**In this episode you'll build:** a free-form two-party conversation between a learner and a mentor.

## What `conversation` does

The `conversation` adapter is a **free-form, two-party** channel:

- Either side may send at any time, in any order.
- It **never auto-closes** — you stop it yourself, or a TTL does.

Both agents' default handlers reply to every inbound message, so once you send the first message the conversation **auto-drives** itself. Your job is to decide when to stop it.

Use it for analyst↔critic exchanges, negotiation, or any genuine dialogue between two specialists.

## Setup

Because `conversation` never stops on its own, we add a helper that polls the WAL and lets us cap the chat at N messages.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

import asyncio

from autogen.beta import Agent
from autogen.beta.config import OpenAIConfig
from autogen.beta.knowledge import MemoryKnowledgeStore
from autogen.beta.network import EV_TEXT, Hub, HubClient, LocalLink, Passport, Resume

config = OpenAIConfig(model="gpt-4.1-mini", temperature=0)


async def wait_for_text_count(hub, channel_id, expected, timeout=180.0):
    """Poll the WAL until `expected` text envelopes exist."""
    deadline = asyncio.get_event_loop().time() + timeout
    while asyncio.get_event_loop().time() < deadline:
        wal = await hub.read_wal(channel_id)
        if sum(1 for e in wal if e.event_type == EV_TEXT) >= expected:
            return
        await asyncio.sleep(0.1)
    raise asyncio.TimeoutError("count not reached")

## A self-driving conversation

We send one opening message, then let the two agents talk. After 4 messages we close the channel ourselves.

In [ ]:
hub = await Hub.open(MemoryKnowledgeStore(), ttl_sweep_interval=0)
link = LocalLink(hub)
learner_hc = HubClient(link, hub=hub)
mentor_hc = HubClient(link, hub=hub)

learner = await learner_hc.register(
    Agent(
        "learner",
        prompt="You are a curious learner. One short sentence per turn.",
        config=config,
    ),
    Passport(name="learner"),
    Resume(),
    attach_plugin=False,
)
mentor = await mentor_hc.register(
    Agent(
        "mentor",
        prompt="You are a patient mentor. One short sentence per turn.",
        config=config,
    ),
    Passport(name="mentor"),
    Resume(),
    attach_plugin=False,
)

# conversation never auto-closes; both handlers reply to every message,
# so the chat drives itself once we send the first line.
channel = await learner.open(type="conversation", target="mentor")
await channel.send(
    "Hi! What is one good first concept to learn in AI?", audience=[mentor.agent_id]
)

# Stop it ourselves after 4 messages.
await wait_for_text_count(hub, channel.channel_id, expected=4)
await channel.close(reason="cap_reached")

names = {learner.agent_id: "learner", mentor.agent_id: "mentor"}
for env in await hub.read_wal(channel.channel_id):
    if env.event_type == EV_TEXT:
        print(f"{names[env.sender_id]}: {env.event_data['text']}")

await learner_hc.close()
await mentor_hc.close()
await hub.close()

## You own the stop signal

The key difference from `consulting`: **nothing closes a `conversation` for you.** That's a feature — the agents would happily talk forever. Three common ways to stop one:

1. **An application-side cap** — count messages on the WAL, then `channel.close()` (what we did).
2. **An empty reply** — the default handler treats an empty body as "say nothing", which ends the chain. Prompt an agent to reply empty when it has nothing to add.
3. **An agent-side close tool** — give an agent a tool that calls `channel.close()` so the LLM decides.

## Additional content

**Same-side sends are allowed.** `conversation` doesn't model turn order, so an agent *can* send twice in a row. If you need strict alternation, that's `discussion`.

**Lenient expectations.** `conversation`'s only default expectation logs a warning after an hour of silence — it won't close the channel on you.

## What's next

Two agents, free-form. Episode 13's `discussion` adapter scales to N agents with a strict round-robin turn order.